# ANN — Predict Disease from Symptoms
**Dataset:** Fake symptom data (created inline — no download)
**Input:** fever, cough, headache | **Output:** Healthy / Cold / Flu

> ANN = Artificial Neural Network. Layers of neurons learn which symptoms match which disease.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# ── Dataset ───────────────────────────────────────────────
# Features: [fever, cough, headache]  Labels: 0=Healthy 1=Cold 2=Flu
X = np.array([
    [0,0,0],[0,0,0],[0,0,0],[0,0,1],[0,1,0],   # Healthy
    [0,1,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],   # Healthy
    [0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],   # Healthy
    [0,1,1],[0,1,0],[0,0,1],[0,1,1],[0,1,0],   # Cold
    [0,1,1],[0,0,1],[0,1,1],[0,1,0],[0,1,1],   # Cold
    [0,1,0],[0,0,1],[0,1,1],[0,1,0],[0,1,0],   # Cold
    [1,1,1],[1,0,1],[1,1,1],[1,1,0],[1,1,1],   # Flu
    [1,1,1],[1,1,1],[1,0,1],[1,1,1],[1,1,1],   # Flu
    [1,1,0],[1,0,1],[1,1,1],[1,1,0],[1,1,1],   # Flu
], dtype='float32')

y = np.array([0]*15 + [1]*15 + [2]*15)

# ── Train / Test Split ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

In [ ]:
# ── Build ANN ─────────────────────────────────────────────
model = Sequential([
    Input(shape=(3,)),                         # 3 symptom features
    Dense(16, activation='relu'),
    Dropout(0.2),                              # drops 20% neurons — prevents overfitting
    Dense(16, activation='relu'),
    Dense(3,  activation='softmax')            # 3 output classes
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=120, batch_size=8,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Train Accuracy: {history.history["accuracy"][-1]*100:.1f}%')
print(f'Test  Accuracy: {history.history["val_accuracy"][-1]*100:.1f}%')

In [ ]:
# ── Accuracy Plot ─────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Test Accuracy', linestyle='--')
plt.title('ANN — Training Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Healthy','Cold','Flu'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — ANN Disease Predictor')
plt.show()

In [ ]:
# ── Predict New Patients ───────────────────────────────────
labels = ['Healthy', 'Cold', 'Flu']

patients = [
    {'name': 'Raj',   'symptoms': [1, 1, 1]},  # fever+cough+headache
    {'name': 'Priya', 'symptoms': [0, 1, 1]},  # cough+headache only
    {'name': 'Sara',  'symptoms': [0, 0, 0]},  # no symptoms
]

print('─' * 45)
for p in patients:
    probs = model.predict(np.array([p['symptoms']]), verbose=0)[0]
    pred  = labels[probs.argmax()]
    conf  = probs.max() * 100
    print(f"{p['name']:6s} | Symptoms {p['symptoms']} → {pred} ({conf:.0f}%)")
print('─' * 45)